# Plan 003.3b — Tag-aware current-file search

A public, synthetic walkthrough of Archiver's bounded tag-aware current-file API and compatible `catalog files` CLI.

Plan 003.3b answers path-level questions such as:

- Which **current files** have all or any requested content tags?
- How do tag filters compose with a path GLOB and existing sort modes?
- How can active tags be displayed without filtering files?
- How do complete totals remain accurate when rows and tag previews are bounded?

The notebook uses only disposable files and removes its temporary catalog and source tree at the end.

## Audience, prerequisites, and learning goals

This tutorial is for developers and users who know Archiver's basic catalog and content-tag concepts.

**Prerequisites**

- Run the notebook from the repository root in the project environment.
- Plan 003.2 concepts: content identity, active assertions, provenance, and retraction.
- Plan 003.3a concepts: multi-tag `all`/`any` matching and bounded tag previews.

**By the end, you will be able to**

1. track how filenames map to content identity and provenance;
2. display active tags on bounded current-file rows without filtering;
3. combine AND/OR tag filters with path GLOBs, provenance, sorting, and limits;
4. distinguish complete matched-file/logical-size totals from bounded rows and previews;
5. use the equivalent `archiver catalog files` CLI with explicit tag-aware options;
6. explain why duplicate paths remain separate file rows while sharing content tags.

## Outline

1. Create and scan disposable synthetic files.
2. Seed content-level user and system tag assertions.
3. Track filenames, simple content IDs, provenance, and tags interactively.
4. Query current files through the typed API.
5. Compose tag filters, path GLOBs, sorting, and bounds.
6. Run the equivalent CLI commands with explicit `!` syntax.
7. Refresh changed content and confirm current-scan isolation.
8. Complete a short exercise and clean up.

## The semantic boundary

One result row represents one **current path** from the requested root's latest successful scan. Tags still describe SHA-256 **content identity**, so duplicate paths remain separate file rows but share the same active tag names.

Requested-tag provenance filters matching only. Displayed tags include every distinct active name on the content, regardless of provenance.

The file limit is applied in SQLite before observations are created. Tag previews are fetched only for content identities present in those bounded file rows; complete matched-file and logical-size totals are computed before the row limit.

## 1. Create disposable synthetic content

Two paths contain identical photo bytes. Another photo, a note, and an untagged binary have distinct content identities. Fixed modification times make the later sort examples deterministic.

In [ ]:
# Import the filesystem, temporary-workspace, and Archiver types used below.
from __future__ import annotations

import os
from pathlib import Path, PurePosixPath
from tempfile import TemporaryDirectory

from archiver import Catalog, TagAwareCurrentFileSearch, TagProvenance

# Create an isolated root so the tutorial never touches real archive data.
workspace = TemporaryDirectory(prefix="archiver-plan-0033b-")
base = Path(workspace.name)
root = base / "demo-root"
(root / "photos").mkdir(parents=True)
(root / "backups").mkdir()
(root / "notes").mkdir()


# Write deterministic bytes and mtimes so scan and sort results are reproducible.
def write_demo(relative_path: str, contents: bytes, mtime_ns: int) -> None:
    path = root / relative_path
    path.write_bytes(contents)
    os.utime(path, ns=(mtime_ns, mtime_ns))


# Reusing these bytes creates two paths with one shared content identity.
shared_bytes = b"same synthetic photo"
write_demo("backups/day-01-copy.jpg", shared_bytes, 1_700_000_001_000_000_000)
write_demo("photos/day-01.jpg", shared_bytes, 1_700_000_002_000_000_000)
write_demo("photos/group.jpg", b"different synthetic group photo", 1_700_000_003_000_000_000)
write_demo("notes/readme.txt", b"synthetic notes", 1_700_000_004_000_000_000)
write_demo("loose.bin", b"untagged bytes", 1_700_000_000_000_000_000)

# Initialize the standard in-root SQLite catalog used by later CLI cells.
control_directory = root / ".archiver"
control_directory.mkdir()
catalog_path = control_directory / "catalog.sqlite"
Catalog.create(catalog_path).close()

print(f"Disposable root: {root}")
for source_path in sorted(path for path in root.rglob("*") if path.is_file() and ".archiver" not in path.parts):
    print(" ", source_path.relative_to(root).as_posix())

## 2. Scan with the CLI

Every CLI example in this notebook uses explicit IPython `!` syntax. The CLI excludes `.archiver` from the scan, and `--no-progress` keeps notebook output compact.

In [ ]:
# Record the disposable root's first successful current state.
!archiver catalog scan "{root}" --no-progress

## 3. Add synthetic user and system assertions

Plan 003.3b is query-only; Plan 003.2 supplies tag mutation. The synthetic provenance below lets us demonstrate that every requested tag must be asserted by the selected provenance under AND matching.

`catalog.content_for_path(root, relative_path)` looks up that path in the root's latest successful scan and returns its `ContentId`. Thus `shared_photo` names the SHA-256 identity of the photo bytes—not the pathname. The duplicate copy resolves to the same identity and automatically shares its tags.

In [ ]:
# Open the catalog and define who produced each synthetic tag assertion.
catalog = Catalog.open(catalog_path)

user = TagProvenance("user", "notebook", "1.0")
system = TagProvenance("system", "demo-classifier", "1.0", "synthetic-rules")

# Resolve current paths to ContentId values: SHA-256 identities for bytes, not paths.
shared_photo = catalog.content_for_path(root, PurePosixPath("photos/day-01.jpg"))
group_photo = catalog.content_for_path(root, PurePosixPath("photos/group.jpg"))
notes = catalog.content_for_path(root, PurePosixPath("notes/readme.txt"))

# Attach assertions to content identity; duplicate paths automatically share them.
for tag in ("family", "favorite"):
    catalog.add_content_tag(shared_photo, tag, user)
for tag in ("family", "picture", "trip:us"):
    catalog.add_content_tag(shared_photo, tag, system)

for tag in ("family", "trip:us"):
    catalog.add_content_tag(group_photo, tag, user)
catalog.add_content_tag(group_photo, "picture", system)

catalog.add_content_tag(notes, "text", user)

print("Shared digest:", shared_photo.digest)
print("Shared paths:", [item.relative_path.as_posix() for item in catalog.find_by_content(root, shared_photo)])

## 4. Track filenames, content identity, provenance, and tags

This `itables` table expands each current filename into its scan-observed modification time and active tag assertions. Simple labels (`C1`, `C2`, ...) make shared content easy to spot: duplicate paths carry the same label because their bytes have the same SHA-256 identity.

Use the global search or per-column header filters to narrow the rows. Click a column heading to sort. An em dash marks content with no active assertion.

`Modified (UTC)` is the mtime stored in the latest successful scan. It describes the observed path and does not participate in content identity.

In [ ]:
# Build one display row per filename/assertion, plus one blank-tag row for untagged content.
from datetime import UTC, datetime

import pandas as pd
from itables import show


def format_mtime_ns(timestamp_ns: int) -> str:
    """Convert a scan-observed nanosecond timestamp to readable UTC."""
    seconds, _ = divmod(timestamp_ns, 1_000_000_000)
    return (
        datetime.fromtimestamp(seconds, tz=UTC)
        .isoformat(timespec="seconds")
        .replace("+00:00", "Z")
    )


observations = catalog.current_files(root)

# Assign short labels in deterministic filename order so shared content is easy to see.
simple_content_ids = {}
for observation in observations:
    if observation.content_id not in simple_content_ids:
        simple_content_ids[observation.content_id] = f"C{len(simple_content_ids) + 1}"

tag_map_rows = []
for observation in observations:
    filename = observation.relative_path.as_posix()
    content_label = simple_content_ids[observation.content_id]
    modified_utc = format_mtime_ns(observation.mtime_ns)
    assertions = catalog.tags_for_content(observation.content_id)

    if not assertions:
        tag_map_rows.append(
            {
                "Filename": filename,
                "Modified (UTC)": modified_utc,
                "Content": content_label,
                "Provenance": "—",
                "Tag": "—",
            }
        )
        continue

    for assertion in assertions:
        provenance = assertion.provenance
        provenance_label = (
            f"{provenance.kind}:{provenance.source_name}@{provenance.source_version}"
        )
        if provenance.source_detail:
            provenance_label = f"{provenance_label} ({provenance.source_detail})"
        tag_map_rows.append(
            {
                "Filename": filename,
                "Modified (UTC)": modified_utc,
                "Content": content_label,
                "Provenance": provenance_label,
                "Tag": assertion.tag,
            }
        )

# Both photo paths should visibly map to the same content label.
shared_labels = {
    row["Content"]
    for row in tag_map_rows
    if row["Filename"] in {"backups/day-01-copy.jpg", "photos/day-01.jpg"}
}
assert len(shared_labels) == 1
assert any(row["Tag"] == "—" for row in tag_map_rows)

tag_map = pd.DataFrame(
    tag_map_rows,
    columns=["Filename", "Modified (UTC)", "Content", "Provenance", "Tag"],
)

In [ ]:
# `connected=False` embeds the table assets, keeping the saved notebook self-contained.
# uses regular expression searches: Tag: (family|picture)
# show(
#     tag_map,
#     searchCols=[
#         {"regex": True, "smart": False}
#         for _ in tag_map.columns
#     ],
#     caption="Current filenames and their active content-tag assertions",
#     showIndex=False,
#     column_filters="header",
#     paging=False,
#     scrollX=True,
#     order=[[0, "asc"], [3, "asc"], [4, "asc"]],
#     connected=True,
# )

## 5. Display tags without filtering files

An empty `tags` sequence is valid. This returns every current path while adding bounded alphabetical active-tag previews. The untagged file remains visible with an empty tag tuple.

The compact helper also prints each scan-observed modification time in UTC, making later date-sort results directly visible.

In [ ]:
# Render complete totals, bounded rows, and tag-preview overflow compactly.
def show_files(label: str, result: TagAwareCurrentFileSearch) -> None:
    print(
        f"{label}: {result.total_file_count} files, "
        f"{result.total_content_count} distinct contents, "
        f"{result.total_file_size_bytes} bytes across file paths"
    )
    for item in result.files:
        observation = item.observation
        overflow = item.active_tag_count - len(item.tags)
        suffix = f" +{overflow}" if overflow else ""
        print(
            f"  {observation.relative_path.as_posix():<28} "
            f"modified={format_mtime_ns(observation.mtime_ns)} "
            f"{observation.content_id.digest[:12]}… size={observation.size_bytes:<3} "
            f"tags={list(item.tags)}{suffix}"
        )


# Empty tags selects every current row and only adds bounded tag previews.
display_only = catalog.search_current_files_with_tags(root, tags=(), limit=20, tag_limit=6)
show_files("all current files", display_only)

# Assertions make the documented result contract executable.
assert display_only.total_file_count == 5
assert display_only.total_content_count == 4
assert display_only.total_file_size_bytes == sum(
    item.observation.size_bytes for item in display_only.files
)
assert len(display_only.files) == 5
assert display_only.files[0].observation.relative_path == PurePosixPath("backups/day-01-copy.jpg")
untagged = next(item for item in display_only.files if item.observation.relative_path == PurePosixPath("loose.bin"))
assert untagged.active_tag_count == 0
assert untagged.tags == ()

## 6. Match all tags or any tag

Repeating a requested tag does not change matching. AND is the default; OR uses `match="any"`. Duplicate paths containing the shared bytes remain separate file rows and reuse the same tag preview.

In [ ]:
# Compare default AND matching with explicit OR matching.
all_result = catalog.search_current_files_with_tags(
    root,
    tags=("family", "trip:us", "family"),
    match="all",
    tag_limit=None,
)
any_result = catalog.search_current_files_with_tags(
    root,
    tags=("favorite", "text"),
    match="any",
    tag_limit=None,
)

show_files("family AND trip:us", all_result)
show_files("favorite OR text", any_result)

assert all_result.total_file_count == 3
assert all_result.total_content_count == 2
assert [item.observation.relative_path.as_posix() for item in all_result.files] == [
    "backups/day-01-copy.jpg",
    "photos/day-01.jpg",
    "photos/group.jpg",
]
assert all_result.files[0].tags == all_result.files[1].tags
assert any_result.total_file_count == 3
assert any_result.total_content_count == 2

## 7. Compose provenance, path GLOB, sorting, and bounds

The system provenance filter below must satisfy both `family` and `picture`. Only the shared photo content qualifies, so both of its current paths match. The user tag `favorite` is still displayed because provenance does not filter the preview.

The second query composes a full-path SQLite GLOB with date sorting. The third deliberately bounds rows and tag names while retaining complete totals.

In [ ]:
# Require both requested tags from system provenance.
system_result = catalog.search_current_files_with_tags(
    root,
    tags=("family", "picture"),
    provenance="system",
    tag_limit=None,
    match="all",
)
show_files("system family AND picture", system_result)

In [ ]:
# Restrict paths, then preserve the existing oldest-first date sort.
photos_by_date = catalog.search_current_files_with_tags(
    root,
    path_glob="photos/*",
    tags=("family",),
    sort_by="date",
    reverse=True,
    tag_limit=3,
)
show_files("family photos, oldest first", photos_by_date)

In [ ]:
# Bound rows and preview tags without changing complete totals.
bounded = catalog.search_current_files_with_tags(
    root,
    tags=("family", "trip:us"),
    limit=2,
    tag_limit=1,
)


show_files("bounded family AND trip:us", bounded)

In [ ]:
(
    bounded.total_file_count,
    bounded.total_content_count,
    bounded.total_file_size_bytes,
)

In [ ]:
assert system_result.total_file_count == 2
assert system_result.total_content_count == 1
assert all("favorite" in item.tags for item in system_result.files)
assert [item.observation.mtime_ns for item in photos_by_date.files] == sorted(
    item.observation.mtime_ns for item in photos_by_date.files
)
assert [item.observation.relative_path.as_posix() for item in photos_by_date.files] == [
    "photos/day-01.jpg",
    "photos/group.jpg",
]
assert bounded.total_file_count == 3
assert bounded.total_content_count == 2
assert len(bounded.files) == 2
assert all(len(item.tags) == 1 for item in bounded.files)

## 8. Use the tag-aware CLI directly

`--show-tags` enables the five-column table without filtering. Tag filters use AND semantics by default, and complete totals remain visible when `--limit` bounds displayed rows.

In [ ]:
# Display active tags for every current file without filtering rows.
!archiver catalog files "{root}" --show-tags

In [ ]:
# Require both tags while bounding file rows and tag names per row.
!archiver catalog files "{root}" --tag family --tag trip:us --limit 2 --display-tag-limit 2

`--provenance` applies independently to every requested tag. `--path` remains the existing case-sensitive SQLite GLOB over complete POSIX relative paths, and all existing sort controls still compose.

In [ ]:
# Compose a path GLOB, provenance filter, and oldest-first date sort.
!archiver catalog files "{root}" --path "photos/*" --tag family --tag picture --provenance system --sort date --reverse

`--match-any-tag` switches requested tags to OR. `--all-tags` fetches every active tag only for the already bounded file rows and wraps long tag lists instead of widening the table without bound.

In [ ]:
# Switch to OR matching and show every active tag on bounded rows.
!archiver catalog files "{root}" --tag favorite --tag text --match-any-tag --all-tags

## 9. Confirm legacy CLI compatibility

With no tag-aware option, `catalog files` dispatches through the original query and renderer. The output intentionally has no `Tags` column.

In [ ]:
# Omit tag options to exercise the unchanged legacy renderer.
!archiver catalog files "{root}" --limit 3 --sort size

## 10. Refresh changed content

Tags belong to content bytes, not pathnames. The notebook explicitly changes the **synthetic** `photos/group.jpg` bytes, then Archiver observes that change during refresh. The old tagged content remains catalog metadata, but it is no longer current at that path.

In [ ]:
# Replace synthetic bytes; the path stays the same but its content identity changes.
write_demo("photos/group.jpg", b"replacement untagged group photo", 1_700_000_005_000_000_000)
print("Synthetic group photo bytes replaced by the notebook.")

In [ ]:
# Refresh the catalog after the synthetic bytes changed.
!archiver catalog scan "{root}" --no-progress

In [ ]:
# Query the latest successful scan; old tagged bytes are no longer at group.jpg.
after_refresh = catalog.search_current_files_with_tags(
    root,
    tags=("family", "trip:us"),
    tag_limit=None,
)
show_files("family AND trip:us after refresh", after_refresh)

assert after_refresh.total_file_count == 2
assert after_refresh.total_content_count == 1
assert [item.observation.relative_path.as_posix() for item in after_refresh.files] == [
    "backups/day-01-copy.jpg",
    "photos/day-01.jpg",
]

The display-only CLI still includes the modified current path, now with no active tags. A filtered command excludes it.

In [ ]:
# Display both current photo paths, including replacement content with no tags.
!archiver catalog files "{root}" --path "photos/*" --show-tags --all-tags

In [ ]:
# Filter out the untagged replacement row.
!archiver catalog files "{root}" --path "photos/*" --tag family --tag trip:us

## Exercise

After the refresh, predict the API result for `family AND picture` with `provenance="system"` and `tag_limit=None`.

- How many current file rows match?
- Why are there two rows but only one content identity?
- Does the user tag `favorite` still appear in each row's displayed tag tuple?

Write your prediction before revealing the answer cell.

In [ ]:
# Run the exercise query and verify the prediction.
exercise = catalog.search_current_files_with_tags(
    root,
    tags=("family", "picture"),
    provenance="system",
    tag_limit=None,
)
show_files("exercise answer", exercise)

assert exercise.total_file_count == 2
assert exercise.total_content_count == 1
assert len({item.observation.content_id for item in exercise.files}) == 1
assert all("favorite" in item.tags for item in exercise.files)

## Pitfalls and extensions

- **Do not collapse duplicate current paths into one file row.** Plan 003.3b returns one row per current path; shared content explains why those rows have identical tags.
- **Do not treat provenance as a display filter.** It qualifies requested tags, while previews include every active tag name.
- **Do not infer totals from bounded rows.** Compare `total_file_count` with `len(files)` and `active_tag_count` with `len(tags)`. `total_content_count` counts shared bytes once, while `total_file_size_bytes` sums every matching path.
- **Do not use `--match-any-tag` or `--provenance` without `--tag`.** The CLI rejects those combinations.
- **Do not expect a tag to follow a pathname after its bytes change.** Tags stay attached to the old content identity.

Optional extension: add a tag to the replacement content and compare `--sort path`, `--sort size`, and `--sort date --reverse` while keeping the same path GLOB.

## Cleanup

In [ ]:
# Close SQLite before deleting the disposable directory and synthetic files.
catalog.close()
workspace.cleanup()
print("Temporary catalog and synthetic files removed.")